In [2]:
from openai import OpenAI
import json

# Point strictly to the local Ollama server
client = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
print("Client initialized. Ensure 'ollama run llama3.1' is active in a separate Windows terminal.")

Client initialized. Ensure 'ollama run llama3.1' is active in a separate Windows terminal.


In [6]:
print("Sending ping to local Llama 3.1 teacher model...")

try:
    response = client.chat.completions.create(
        model="llama3.1", 
        messages=[
            {"role": "system", "content": "You output strict JSON. Respond with a JSON object containing a key 'status' and value 'operational'."},
            {"role": "user", "content": "Are you online?"}
        ],
        response_format={"type": "json_object"},
        temperature=0.1
    )
    
    output = response.choices[0].message.content
    print(f"Raw Output:\n{output}")
    
    # Validate JSON parsing capability
    parsed = json.loads(output)
    print("\nSuccess, Master. Llama 3.1 is operational and formatting JSON correctly.")
    
except Exception as e:
    print(f"\nConnection or Parsing Failure: {e}")

Sending ping to local Llama 3.1 teacher model...
Raw Output:
{
  "status": "operational"
}

Success, Master. Llama 3.1 is operational and formatting JSON correctly.


In [3]:
import fitz  # PyMuPDF
import json
import os
from tqdm.auto import tqdm # Auto-detects Jupyter environment for better progress bars

print("Extraction libraries successfully loaded.")

Extraction libraries successfully loaded.


c:\Users\Vishwas\miniconda3\envs\dataset\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
def extract_pdf_to_json(pdf_path, output_json):
    """
    Extracts text page-by-page from a PDF, stripping empty pages, 
    and packages it with metadata for the Teacher LLM.
    """
    if not os.path.exists(pdf_path):
        print(f"Error: The file '{pdf_path}' was not found.")
        print("Please ensure the PDF is in the same directory as this notebook.")
        return
    
    print(f"Opening '{pdf_path}'...")
    doc = fitz.open(pdf_path)
    metadata = doc.metadata
    extracted_data = []
    
    print(f"Scanning {len(doc)} pages...")
    for page_num in tqdm(range(len(doc)), desc="Extraction Progress"):
        page = doc.load_page(page_num)
        text = page.get_text("text").strip()
        
        # Skip purely blank pages or pages with minimal text (like just a page number)
        if len(text) < 50:
            continue
            
        extracted_data.append({
            "page_number": page_num + 1,
            "source_file": os.path.basename(pdf_path),
            "title": metadata.get("title", "Unknown Document"),
            "text": text
        })
        
    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(extracted_data, f, indent=4)
        
    print(f"Extraction complete. Successfully saved {len(extracted_data)} populated pages to '{output_json}'.")

In [ ]:
# --- CONFIGURATION ---
# Replace this string with the exact name of your PDF file
INPUT_PDF = "text.pdf" 
OUTPUT_JSON = "raw_extracted_text.json"

# Run the extractor
extract_pdf_to_json(INPUT_PDF, OUTPUT_JSON)

# Verify the output file was created
if os.path.exists(OUTPUT_JSON):
    file_size = os.path.getsize(OUTPUT_JSON) / 1024 # Size in KB
    print(f"\nVerification: '{OUTPUT_JSON}' generated successfully ({file_size:.2f} KB).")

Opening 'text.pdf'...
Scanning 1440 pages...


Extraction Progress: 100%|██████████| 1440/1440 [00:00<00:00, 2560.91it/s]


Extraction complete. Successfully saved 1412 populated pages to 'raw_extracted_text.json'.

Verification: 'raw_extracted_text.json' generated successfully (1330.99 KB).


In [1]:
import json
from openai import OpenAI
from tqdm.auto import tqdm

# Initialize connection to your local Llama 3.1
client = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

def clean_json_response(raw_text):
    """
    Local models occasionally hallucinate markdown backticks around JSON outputs.
    This function rigorously strips them to prevent JSONDecodeErrors.
    """
    text = raw_text.strip()
    if text.startswith("```json"):
        text = text[7:]
    if text.startswith("```"):
        text = text[3:]
    if text.endswith("```"):
        text = text[:-3]
    return text.strip()

def chunk_text(text, chunk_size=800, overlap=150):
    """
    Splits text into sliding windows. 
    overlap=150 ensures context is not lost if a critical survival step 
    is cut precisely at the chunk boundary.
    """
    words = text.split()
    chunks = [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size - overlap)]
    return chunks

def generate_pairs(text_chunk):
    """
    Forces Llama 3.1 to generate exact ChatML formatted Q&A pairs.
    """
    system_prompt = """
    You are an expert data annotator. Extract survival knowledge from the provided text. 
    Generate 5 highly specific instruction-response pairs based STRICTLY on the text.
    Return ONLY a JSON array of objects fitting this EXACT ChatML schema. Do not include extra conversational text.
    {
      "datasets": [
        {
          "messages": [
            {"role": "system", "content": "You are a localized survival expert operating completely offline. Provide concise, step-by-step guidance."},
            {"role": "user", "content": "<Specific question derived from text>"},
            {"role": "assistant", "content": "<Specific, step-by-step answer derived from text>"}
          ]
        }
      ]
    }
    """
    try:
        response = client.chat.completions.create(
            model="llama3.1",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"TEXT:\n{text_chunk}"}
            ],
            response_format={"type": "json_object"},
            temperature=0.1 # Strictly 0.1 to prevent creative hallucinations
        )
        
        raw_output = response.choices[0].message.content
        clean_output = clean_json_response(raw_output)
        result = json.loads(clean_output)
        
        # Safely extract the array, defaulting to an empty list if the model deviated
        return result.get("datasets", [])
        
    except Exception as e:
        # If the model fails a chunk, we log it and move on without crashing the loop
        # print(f"Skipping chunk due to generation error: {e}") 
        return []

print("Generation functions defined and ready.")

Generation functions defined and ready.


c:\Users\Vishwas\miniconda3\envs\dataset\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# --- CONFIGURATION ---
INPUT_JSON = "raw_extracted_text.json"
OUTPUT_JSONL = "survival_dataset.jsonl"

print(f"Loading extracted text from '{INPUT_JSON}'...")

try:
    with open(INPUT_JSON, 'r', encoding='utf-8') as f:
        pages = json.load(f)
except FileNotFoundError:
    print(f"Error: Master, '{INPUT_JSON}' does not exist. Please run the PyMuPDF cell first.")
    pages = []

if pages:
    print(f"Initiating synthetic generation for {len(pages)} pages using Llama 3.1...")
    
    # Open the file in 'append' mode ('a') so we stream results to disk immediately
    with open(OUTPUT_JSONL, 'a', encoding='utf-8') as outfile:
        
        for page in tqdm(pages, desc="Processing Pages"):
            chunks = chunk_text(page["text"])
            
            for chunk in chunks:
                qa_data = generate_pairs(chunk)
                
                for conversation in qa_data:
                    # Dump each conversation as a single, un-indented JSON line
                    json.dump(conversation, outfile)
                    outfile.write('\n')
                    
    print(f"\nDistillation complete. Synthetic dataset saved to '{OUTPUT_JSONL}'.")

Loading extracted text from 'raw_extracted_text.json'...
Initiating synthetic generation for 1412 pages using Llama 3.1...


Processing Pages:   0%|          | 0/1412 [00:00<?, ?it/s]

In [3]:
# Extract PDF to structured markdown
def extract_pdf_to_markdown(pdf_path, output_md):
    """
    Extracts text from PDF with hierarchical structure and saves as markdown.
    """
    if not os.path.exists(pdf_path):
        print(f"Error: The file '{pdf_path}' was not found.")
        return
    
    print(f"Opening '{pdf_path}'...")
    doc = fitz.open(pdf_path)
    
    with open(output_md, 'w', encoding='utf-8') as f:
        # Add title from metadata
        title = doc.metadata.get("title", "Document")
        f.write(f"# {title}\n\n")
        
        print(f"Extracting {len(doc)} pages to markdown...")
        for page_num in tqdm(range(len(doc)), desc="Extraction Progress"):
            page = doc.load_page(page_num)
            text = page.get_text("text").strip()
            
            if len(text) < 50:
                continue
            
            # Add page break as section
            f.write(f"## Page {page_num + 1}\n\n")
            f.write(text + "\n\n")
    
    print(f"Markdown export complete. Saved to '{output_md}'.")

# Run extraction
extract_pdf_to_markdown("rulebook.pdf", "rulebook.md")

Opening 'rulebook.pdf'...
Extracting 8 pages to markdown...


Extraction Progress: 100%|██████████| 8/8 [00:00<00:00, 377.64it/s]

Markdown export complete. Saved to 'rulebook.md'.
